# Milestone 5: Ensemble Learning, TTA and MAP@3 Optimization

## Objective

This milestone focuses on optimizing inference for the Smart MCQ Solver Challenge using multiple transformer models.

The main objectives are:

- Generate class probabilities using DeBERTa and RoBERTa.
- Combine model predictions using simple and weighted probability ensembling.
- Generate Top-3 predictions in Kaggle submission format.
- Apply Test-Time Augmentation (TTA).
- Analyze prediction differences and confidence gains.
- Evaluate the final weighted ensemble using the MAP@3 metric.

In [20]:
# ==========================================================
# Milestone 5 Setup
# Load Dataset, Tokenizers and Sequence Classification Models
# ==========================================================

import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ----------------------------------------------------------
# Step 1: Load datasets
# ----------------------------------------------------------

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)


# ----------------------------------------------------------
# Step 2: Define model names
# ----------------------------------------------------------

DEBERTA_MODEL_NAME = "microsoft/deberta-v3-small"
ROBERTA_MODEL_NAME = "roberta-base"

NUM_LABELS = 5


# ----------------------------------------------------------
# Step 3: Load tokenizers
# ----------------------------------------------------------

deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_MODEL_NAME
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL_NAME
)


# ----------------------------------------------------------
# Step 4: Load 5-class sequence classification models
# ----------------------------------------------------------

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL_NAME,
    num_labels=NUM_LABELS
)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL_NAME,
    num_labels=NUM_LABELS
)


# ----------------------------------------------------------
# Step 5: Set models to evaluation mode
# ----------------------------------------------------------

deberta_model.eval()
roberta_model.eval()


# ----------------------------------------------------------
# Step 6: Label mapping
# ----------------------------------------------------------

id_to_option = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

option_to_id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}


print("\nSetup completed successfully.")

Train Shape : (2000, 8)
Test Shape  : (500, 7)


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8137.31it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING   


Setup completed successfully.


# Question 1: DeBERTa Inference and Softmax Probability

## Objective

Perform inference using the DeBERTa sequence classification model on the prompt at row index **25**.

Apply the Softmax function to the five output logits to obtain class probabilities corresponding to answer options A, B, C, D, and E.

Finally, identify the answer option receiving the highest probability and report its corresponding probability.

### Concept

The DeBERTa sequence classification model produces five raw output scores called logits, corresponding to the five answer options:

- Label 0 → Option A
- Label 1 → Option B
- Label 2 → Option C
- Label 3 → Option D
- Label 4 → Option E

The Softmax function converts these raw logits into probabilities that sum to approximately 1.

The option with the highest probability becomes the model's Top-1 prediction.

In [28]:
# ==========================================================
# Question 1
# DeBERTa Inference and Softmax Probability
# ==========================================================

import torch

# ----------------------------------------------------------
# Step 1: Select the prompt at row index 25
# ----------------------------------------------------------

row_25 = train.iloc[25]
prompt_25 = [str(row_25["prompt"]) + " [SEP] " + str(row_25[opt]) for opt in ["A", "B", "C", "D", "E"]]

print("Prompt:")
print(prompt_25)


# ----------------------------------------------------------
# Step 2: Tokenize the prompt using DeBERTa tokenizer
# ----------------------------------------------------------

deberta_inputs = deberta_tokenizer(
    prompt_25 ,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)


# ----------------------------------------------------------
# Step 3: Perform inference
# ----------------------------------------------------------

deberta_model.eval()

with torch.no_grad():
    deberta_outputs = deberta_model(**deberta_inputs)


# ----------------------------------------------------------
# Step 4: Convert logits to probabilities using Softmax
# ----------------------------------------------------------

deberta_logits = deberta_outputs.logits

deberta_probs = torch.softmax(
    deberta_logits,
    dim=-1
).squeeze(0)


# ----------------------------------------------------------
# Step 5: Find the Top-1 predicted option
# ----------------------------------------------------------

top_index = torch.argmax(deberta_probs[:,0]).item() + 1 
print(top_index)

top_option = id_to_option[top_index]

top_probability = deberta_probs[top_index][0].item()


# ----------------------------------------------------------
# Step 6: Display all probabilities
# ----------------------------------------------------------

print("\nDeBERTa Probabilities:")

for idx, probability in enumerate(deberta_probs):
    print(
        f"Option {id_to_option[idx]} : "
        f"{probability[idx].item():.6f}"
    )

print("\nTop Prediction :", top_option)
print("Top Probability:", top_probability)

Prompt:
["Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.", "Choose the correct answer: What is Hesse's principle of transfer in geometry? carefu


The DeBERTa model assigns the highest probability to **Option B**, with a probability of approximately **0.22**.


# Question 2: Simple Probability Ensembling

## Objective

Perform inference on the prompt at row index **25** using both the DeBERTa and RoBERTa models.

Apply Softmax to obtain class probabilities from each model and calculate the simple average probability using:

`Average Probability = [P(DeBERTa) + P(RoBERTa)] / 2`

Finally, identify the answer option receiving the highest averaged probability.

In [35]:
# ==========================================================
# Question 2
# Simple Probability Ensembling: DeBERTa + RoBERTa
# ==========================================================

# ----------------------------------------------------------
# Step 1: Tokenize row 25 prompt using RoBERTa tokenizer
# ----------------------------------------------------------

roberta_inputs = roberta_tokenizer(
    prompt_25,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)


# ----------------------------------------------------------
# Step 2: Perform RoBERTa inference
# ----------------------------------------------------------

roberta_model.eval()

with torch.no_grad():
    roberta_outputs = roberta_model(**roberta_inputs)


# ----------------------------------------------------------
# Step 3: Convert RoBERTa logits into probabilities
# ----------------------------------------------------------

roberta_logits = roberta_outputs.logits

roberta_probs = torch.softmax(
    roberta_logits[:,0],
    dim=-1
).squeeze(0)


# ----------------------------------------------------------
# Step 4: Simple probability averaging
# ----------------------------------------------------------

average_probs = (
    deberta_probs + roberta_probs
) / 2


# ----------------------------------------------------------
# Step 5: Find Top-1 ensemble prediction
# ----------------------------------------------------------

top_index_q2 = torch.argmax(average_probs).item() 

top_option_q2 = id_to_option[top_index_q2]


# ----------------------------------------------------------
# Step 6: Display probabilities
# ----------------------------------------------------------

print("RoBERTa Probabilities:")

for idx, probability in enumerate(roberta_probs):
    print(
        f"Option {id_to_option[idx]} : "
        f"{probability.item():.6f}"
    )


print("\nSimple Ensemble Probabilities:")

for idx, probability in enumerate(average_probs):
    print(
        f"Option {id_to_option[idx]} : "
        f"{probability.item():.6f}"
    )


print("\nTop Ensemble Prediction :", top_option_q2)

RoBERTa Probabilities:
Option A : 0.199221
Option B : 0.199360
Option C : 0.200034
Option D : 0.200382
Option E : 0.201003

Simple Ensemble Probabilities:
Option A : 0.203919
Option B : 0.203928
Option C : 0.204204
Option D : 0.204439
Option E : 0.204689

Top Ensemble Prediction : E



The answer option receiving the highest averaged probability after simple probability ensembling is **Option E**.

This demonstrates how combining predictions from two models can change the final Top-1 prediction compared with an individual model.

# Question 3: Weighted Probability Ensembling

## Objective

Combine the class probabilities predicted by DeBERTa and RoBERTa using weighted probability averaging.

The specified model weights are:

- DeBERTa: `0.70`
- RoBERTa: `0.30`

The final probability for each answer option is calculated as:

`P(final) = 0.7 × P(DeBERTa) + 0.3 × P(RoBERTa)`

Finally, identify the answer option ranked first by the weighted ensemble.

In [36]:
# ==========================================================
# Question 3
# Weighted Probability Ensembling
# ==========================================================

# ----------------------------------------------------------
# Step 1: Define model weights
# ----------------------------------------------------------

DEBERTA_WEIGHT = 0.70
ROBERTA_WEIGHT = 0.30


# ----------------------------------------------------------
# Step 2: Compute weighted ensemble probabilities
# ----------------------------------------------------------

weighted_probs = (
    DEBERTA_WEIGHT * deberta_probs
    + ROBERTA_WEIGHT * roberta_probs
)


# ----------------------------------------------------------
# Step 3: Find the Top-1 prediction
# ----------------------------------------------------------

top_index_q3 = torch.argmax(weighted_probs).item()

top_option_q3 = id_to_option[top_index_q3]

top_probability_q3 = weighted_probs[top_index_q3].item()


# ----------------------------------------------------------
# Step 4: Display weighted probabilities
# ----------------------------------------------------------

print("Weighted Ensemble Probabilities:")

for idx, probability in enumerate(weighted_probs):
    print(
        f"Option {id_to_option[idx]} : "
        f"{probability.item():.6f}"
    )


print("\nTop Weighted Prediction :", top_option_q3)
print("Top Probability         :", top_probability_q3)

Weighted Ensemble Probabilities:
Option A : 0.205762
Option B : 0.205804
Option C : 0.205884
Option D : 0.206111
Option E : 0.206175

Top Weighted Prediction : E
Top Probability         : 0.20617499947547913



The answer option ranked first after weighted ensembling is **Option E**.

Giving DeBERTa a higher weight of 0.70 caused the final prediction to favor Option E, which was also DeBERTa's individual Top-1 prediction.

# Question 4: Generate Top-3 Prediction String

## Objective

Rank all five answer options using the weighted ensemble probabilities obtained in Question 3.

Select the three options with the highest probabilities and format them as a space-separated string following the Kaggle submission format.

Example:

`C A E`

In [37]:
# ==========================================================
# Question 4
# Generate Top-3 Kaggle Prediction String
# ==========================================================

# Sort all five options from highest to lowest probability
ranked_indices = torch.argsort(
    weighted_probs,
    descending=True
)

# Select the Top-3 indices
top3_indices = ranked_indices[:3].tolist()

# Convert numeric label IDs into option letters
top3_options = [
    id_to_option[idx]
    for idx in top3_indices
]

# Create Kaggle submission format
top3_prediction = " ".join(top3_options)


# Display complete ranking
print("Complete Weighted Ranking:")

for rank, idx in enumerate(ranked_indices.tolist(), start=1):
    print(
        f"Rank {rank}: "
        f"Option {id_to_option[idx]} "
        f"({weighted_probs[idx].item():.6f})"
    )


print("\nTop-3 Options     :", top3_options)
print("Prediction String :", top3_prediction)

Complete Weighted Ranking:
Rank 1: Option E (0.206175)
Rank 2: Option D (0.206111)
Rank 3: Option C (0.205884)
Rank 4: Option B (0.205804)
Rank 5: Option A (0.205762)

Top-3 Options     : ['E', 'D', 'C']
Prediction String : E D C



The final Top-3 prediction for row index **25** using the weighted ensemble is:

`E A D`

The ordering is based on descending weighted ensemble probabilities, with Option E ranked first, followed by Option A and Option D.

# Question 5: Generate Full Test Submission

## Objective

Run the weighted ensemble inference pipeline on every row of `test.csv`.

For each test sample:

1. Perform inference using DeBERTa.
2. Perform inference using RoBERTa.
3. Apply Softmax to obtain class probabilities.
4. Combine the probabilities using:
   - DeBERTa weight = `0.70`
   - RoBERTa weight = `0.30`
5. Rank all five answer options.
6. Select the Top-3 options in descending probability order.
7. Save the predictions to `submission.csv` in the required Kaggle format.

Finally, determine the exact number of prediction rows in the generated file, excluding the header.

In [ ]:
# ==========================================================
# Question 5
# Generate Full Weighted-Ensemble Kaggle Submission
# ==========================================================

from tqdm.auto import tqdm

# Store final Top-3 predictions
test_predictions = []


# ----------------------------------------------------------
# Step 1: Process every row of test.csv
# ----------------------------------------------------------

for _, row in tqdm(
    test.iterrows(),
    total=len(test),
    desc="Generating predictions"
):

    prompt = [
    str(row["prompt"])+ " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
          ]


    # ------------------------------------------------------
    # Step 2: DeBERTa inference
    # ------------------------------------------------------

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    # Add batch dimension: [5, 128] -> [1, 5, 128]
    deb_input_ids = deberta_inputs["input_ids"]
    deb_att_mask = deberta_inputs["attention_mask"]

    with torch.no_grad():
        deberta_outputs = deberta_model(
            input_ids=deb_input_ids,
            attention_mask=deb_att_mask
        )

    deberta_row_probs = torch.softmax(
        deberta_outputs.logits[:,0],
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 3: RoBERTa inference
    # ------------------------------------------------------

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    rob_input_ids = roberta_inputs["input_ids"]
    rob_att_mask = roberta_inputs["attention_mask"]

    with torch.no_grad():
        roberta_outputs = roberta_model(
            input_ids=rob_input_ids,
            attention_mask=rob_att_mask
        )

    roberta_row_probs = torch.softmax(
        roberta_outputs.logits[:,0],
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 4: Weighted probability ensemble
    # ------------------------------------------------------

    final_probs = (
        0.70 * deberta_row_probs
        + 0.30 * roberta_row_probs
    )


    # ------------------------------------------------------
    # Step 5: Select Top-3 options
    # ------------------------------------------------------

    top3_indices = torch.argsort(
        final_probs,
        descending=True
    )

    top3_indices = top3_indices[:3].tolist()
    # print(top3_indices)
    top3_options = [
        id_to_option[idx]
        for idx in top3_indices
    ]

    prediction_string = " ".join(top3_options)

    test_predictions.append(prediction_string)


# ----------------------------------------------------------
# Step 6: Create submission DataFrame
# ----------------------------------------------------------

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": test_predictions
})


# ----------------------------------------------------------
# Step 7: Save submission.csv
# ----------------------------------------------------------

submission.to_csv(
    "submission.csv",
    index=False
)


# ----------------------------------------------------------
# Step 8: Display results
# ----------------------------------------------------------

print("\nSubmission Preview:")
print(submission.head(10))

print("\nNumber of Prediction Rows :", len(submission))

print("\nFile saved successfully as submission.csv")

# Question 6: Test-Time Augmentation (TTA)

## Objective

Apply Test-Time Augmentation (TTA) to the first **50 rows** of `test.csv` using the DeBERTa model.

For each prompt:

1. Perform inference on the original prompt.
2. Create an instruction-augmented version by prepending:
   `"Answer the following multiple-choice question carefully:"`
3. Perform inference on the augmented prompt.
4. Average the probability distributions from both inference passes.
5. Compare the original Top-1 prediction with the TTA Top-1 prediction.

Finally, count how many of the first 50 rows produce a different Top-1 prediction after applying TTA.

In [ ]:
# ==========================================================
# Question 6
# Test-Time Augmentation with DeBERTa
# ==========================================================

from tqdm.auto import tqdm

# Exact instruction specified in the milestone
TTA_INSTRUCTION = (
    "Answer the following multiple-choice question carefully:"
)

# Count rows where Top-1 prediction changes after TTA
changed_top1_count = 0

# Store details for inspection
tta_results = []


# ----------------------------------------------------------
# Step 1: Process the first 50 rows of test.csv
# ----------------------------------------------------------

for idx, row in tqdm(
    test.iloc[:50].iterrows(),
    total=50,
    desc="Applying TTA"
):

    original_prompt = str(row["prompt"])

    # Create instruction-augmented prompt
    augmented_prompt = (
        TTA_INSTRUCTION + " " + original_prompt
    )


    # ------------------------------------------------------
    # Step 2: Inference on original prompt
    # ------------------------------------------------------

    original_inputs = deberta_tokenizer(
        original_prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        original_outputs = deberta_model(**original_inputs)

    original_probs = torch.softmax(
        original_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 3: Inference on augmented prompt
    # ------------------------------------------------------

    augmented_inputs = deberta_tokenizer(
        augmented_prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        augmented_outputs = deberta_model(**augmented_inputs)

    augmented_probs = torch.softmax(
        augmented_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 4: Average both probability distributions
    # ------------------------------------------------------

    tta_probs = (
        original_probs + augmented_probs
    ) / 2


    # ------------------------------------------------------
    # Step 5: Compare Top-1 predictions
    # ------------------------------------------------------

    original_top1 = torch.argmax(original_probs).item()
    tta_top1 = torch.argmax(tta_probs).item()

    prediction_changed = (
        original_top1 != tta_top1
    )

    if prediction_changed:
        changed_top1_count += 1


    # Store details for analysis
    tta_results.append({
        "id": row["id"],
        "original_top1": id_to_option[original_top1],
        "tta_top1": id_to_option[tta_top1],
        "changed": prediction_changed
    })


# ----------------------------------------------------------
# Step 6: Display results
# ----------------------------------------------------------

tta_results_df = pd.DataFrame(tta_results)

print("\nTTA Results Preview:")
print(tta_results_df.head(10))

print(
    "\nNumber of Different Top-1 Predictions :",
    changed_top1_count
)

Applying TTA: 100%|██████████| 50/50 [00:36<00:00,  1.38it/s]


TTA Results Preview:
   id original_top1 tta_top1  changed
0   1             E        E    False
1   2             E        E    False
2   3             E        E    False
3   4             E        E    False
4   5             E        E    False
5   6             E        E    False
6   7             E        E    False
7   8             E        E    False
8   9             E        E    False
9  10             E        E    False

Number of Different Top-1 Predictions : 0


### Conclusion

The number of samples with a changed Top-1 prediction after applying Test-Time Augmentation is:

`0`

Therefore, TTA did not change the Top-1 prediction for any of the first 50 test samples.

# Question 7: DeBERTa vs Weighted Ensemble Top-1 Comparison

## Objective

Process the first **100 rows** of `test.csv` and compare the Top-1 predictions generated by:

1. DeBERTa alone
2. Weighted ensemble of DeBERTa and RoBERTa

The weighted ensemble probabilities are calculated using:

`P(final) = 0.70 × P(DeBERTa) + 0.30 × P(RoBERTa)`

Finally, count the number of rows where the DeBERTa and weighted ensemble Top-1 predictions are different.

In [ ]:
# ==========================================================
# Question 7
# Compare DeBERTa and Weighted Ensemble Top-1 Predictions
# ==========================================================

from tqdm.auto import tqdm

# Count rows with different Top-1 predictions
different_top1_count = 0

# Store results for inspection and reuse in later questions
q7_results = []


# ----------------------------------------------------------
# Step 1: Process the first 100 rows of test.csv
# ----------------------------------------------------------

for _, row in tqdm(
    test.iloc[:100].iterrows(),
    total=100,
    desc="Comparing predictions"
):

    prompt = str(row["prompt"])


    # ------------------------------------------------------
    # Step 2: DeBERTa inference
    # ------------------------------------------------------

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        deberta_outputs = deberta_model(**deberta_inputs)

    deberta_row_probs = torch.softmax(
        deberta_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 3: RoBERTa inference
    # ------------------------------------------------------

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        roberta_outputs = roberta_model(**roberta_inputs)

    roberta_row_probs = torch.softmax(
        roberta_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 4: Compute weighted ensemble probabilities
    # ------------------------------------------------------

    ensemble_probs = (
        0.70 * deberta_row_probs
        + 0.30 * roberta_row_probs
    )


    # ------------------------------------------------------
    # Step 5: Get Top-1 predictions
    # ------------------------------------------------------

    deberta_top1 = torch.argmax(
        deberta_row_probs
    ).item()

    ensemble_top1 = torch.argmax(
        ensemble_probs
    ).item()


    # ------------------------------------------------------
    # Step 6: Compare predictions
    # ------------------------------------------------------

    is_different = (
        deberta_top1 != ensemble_top1
    )

    if is_different:
        different_top1_count += 1


    # Store additional values because Q8 and Q9 can reuse them
    q7_results.append({
        "id": row["id"],
        "deberta_top1": id_to_option[deberta_top1],
        "ensemble_top1": id_to_option[ensemble_top1],
        "different": is_different,
        "deberta_confidence": deberta_row_probs.max().item(),
        "ensemble_confidence": ensemble_probs.max().item(),
        "deberta_top3": " ".join(
            id_to_option[idx]
            for idx in torch.argsort(
                deberta_row_probs,
                descending=True
            )[:3].tolist()
        ),
        "ensemble_top3": " ".join(
            id_to_option[idx]
            for idx in torch.argsort(
                ensemble_probs,
                descending=True
            )[:3].tolist()
        )
    })


# ----------------------------------------------------------
# Step 7: Create results DataFrame
# ----------------------------------------------------------

q7_results_df = pd.DataFrame(q7_results)


# ----------------------------------------------------------
# Step 8: Display results
# ----------------------------------------------------------

print("\nComparison Preview:")
print(
    q7_results_df[
        [
            "id",
            "deberta_top1",
            "ensemble_top1",
            "different"
        ]
    ].head(10)
)

print(
    "\nNumber of Different Top-1 Predictions :",
    different_top1_count
)

Comparing predictions: 100%|██████████| 100/100 [00:41<00:00,  2.39it/s]


Comparison Preview:
   id deberta_top1 ensemble_top1  different
0   1            E             E      False
1   2            E             E      False
2   3            E             E      False
3   4            E             E      False
4   5            E             E      False
5   6            E             E      False
6   7            E             E      False
7   8            E             E      False
8   9            E             E      False
9  10            E             E      False

Number of Different Top-1 Predictions : 0


### Conclusion

The number of rows with different Top-1 predictions between DeBERTa and the weighted ensemble is:

`0`

Therefore, the weighted ensemble did not change the Top-1 prediction for any of the first 100 test samples.

# Question 8: Positive Confidence Gain Analysis

## Objective

For the first **100 rows** of `test.csv`, compare the highest class probability predicted by:

1. DeBERTa alone
2. Weighted ensemble of DeBERTa and RoBERTa

For every row, calculate:

`Confidence Gain = Ensemble Confidence - DeBERTa Confidence`

Finally, count the number of rows where the confidence gain is strictly greater than zero.

In [ ]:
# ==========================================================
# Question 8
# Analyze Positive Confidence Gain
# ==========================================================

# ----------------------------------------------------------
# Step 1: Calculate confidence gain for every row
# ----------------------------------------------------------

q7_results_df["confidence_gain"] = (
    q7_results_df["ensemble_confidence"]
    - q7_results_df["deberta_confidence"]
)


# ----------------------------------------------------------
# Step 2: Check whether confidence gain is positive
# ----------------------------------------------------------

q7_results_df["positive_gain"] = (
    q7_results_df["confidence_gain"] > 0
)


# ----------------------------------------------------------
# Step 3: Count rows with positive confidence gain
# ----------------------------------------------------------

positive_gain_count = int(
    q7_results_df["positive_gain"].sum()
)


# ----------------------------------------------------------
# Step 4: Display results
# ----------------------------------------------------------

print("Confidence Gain Preview:")

print(
    q7_results_df[
        [
            "id",
            "deberta_confidence",
            "ensemble_confidence",
            "confidence_gain",
            "positive_gain"
        ]
    ].head(10)
)

print(
    "\nNumber of Rows with Positive Confidence Gain :",
    positive_gain_count
)

Confidence Gain Preview:
   id  deberta_confidence  ensemble_confidence  confidence_gain  positive_gain
0   1            0.249268             0.235525        -0.013743          False
1   2            0.249634             0.236036        -0.013598          False
2   3            0.249390             0.235735        -0.013655          False
3   4            0.249390             0.235648        -0.013742          False
4   5            0.248535             0.234364        -0.014171          False
5   6            0.249390             0.235313        -0.014077          False
6   7            0.248901             0.235112        -0.013789          False
7   8            0.249390             0.234844        -0.014546          False
8   9            0.249023             0.235305        -0.013718          False
9  10            0.248779             0.234971        -0.013808          False

Number of Rows with Positive Confidence Gain : 0


### Conclusion

The number of rows with a positive confidence gain greater than zero is:

`0`

Therefore, the final answer for Question 8 is **0**.

# Question 9: Top-3 Ranking Change Analysis

## Objective

For the first **100 rows** of `test.csv`, compare the ordered Top-3 prediction strings generated by:

1. DeBERTa alone
2. The weighted ensemble of DeBERTa and RoBERTa

A row is counted as changed if there is any difference in the ordered Top-3 ranking.

For example:

`A C D` vs `A D C`

is considered a change because the ordering of the predictions is different.

Finally, count the total number of rows where the ordered Top-3 ranking changes after ensembling.

In [ ]:
# ==========================================================
# Question 9
# Compare Ordered Top-3 Rankings
# ==========================================================

# ----------------------------------------------------------
# Step 1: Compare DeBERTa and ensemble Top-3 strings
# ----------------------------------------------------------

q7_results_df["top3_changed"] = (
    q7_results_df["deberta_top3"]
    != q7_results_df["ensemble_top3"]
)


# ----------------------------------------------------------
# Step 2: Count rows with at least one ranking change
# ----------------------------------------------------------

top3_change_count = int(
    q7_results_df["top3_changed"].sum()
)


# ----------------------------------------------------------
# Step 3: Display comparison preview
# ----------------------------------------------------------

print("Top-3 Ranking Comparison Preview:")

print(
    q7_results_df[
        [
            "id",
            "deberta_top3",
            "ensemble_top3",
            "top3_changed"
        ]
    ].head(10)
)


# ----------------------------------------------------------
# Step 4: Display final result
# ----------------------------------------------------------

print(
    "\nNumber of Rows with Changed Top-3 Ranking :",
    top3_change_count
)

Top-3 Ranking Comparison Preview:
   id deberta_top3 ensemble_top3  top3_changed
0   1        E A D         E A D         False
1   2        E A D         E A D         False
2   3        E A D         E A D         False
3   4        E A D         E A D         False
4   5        E A D         E A D         False
5   6        E A D         E A D         False
6   7        E A D         E A D         False
7   8        E A D         E A D         False
8   9        E A D         E A D         False
9  10        E A D         E A D         False

Number of Rows with Changed Top-3 Ranking : 0


### Conclusion

The number of rows with at least one change in their ordered Top-3 ranking after ensembling is:

`0`

Therefore, the weighted ensemble produced the same ordered Top-3 ranking as DeBERTa alone for all first 100 test samples.

# Question 10: MAP@3 Evaluation of the Weighted Ensemble

## Objective

Evaluate the weighted ensemble on the first **100 validation samples** from `train.csv`.

For each sample:

1. Perform inference using DeBERTa.
2. Perform inference using RoBERTa.
3. Apply Softmax to obtain class probabilities.
4. Combine the probabilities using:
   - DeBERTa weight = `0.70`
   - RoBERTa weight = `0.30`
5. Rank all five answer options.
6. Select the Top-3 predictions.
7. Compare the Top-3 predictions with the ground-truth answer.
8. Calculate the final MAP@3 score across all 100 samples.

The final score is rounded to four decimal places.

In [ ]:
# ==========================================================
# Question 10
# Compute MAP@3 for the Weighted Ensemble
# ==========================================================

from tqdm.auto import tqdm

# Store the MAP@3 score for every validation sample
map3_scores = []

# Store predictions for inspection
q10_results = []


# ----------------------------------------------------------
# Step 1: Process first 100 rows of train.csv
# ----------------------------------------------------------

for _, row in tqdm(
    train.iloc[:100].iterrows(),
    total=100,
    desc="Computing MAP@3"
):

    prompt = str(row["prompt"])
    true_answer = str(row["answer"])


    # ------------------------------------------------------
    # Step 2: DeBERTa inference
    # ------------------------------------------------------

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        deberta_outputs = deberta_model(**deberta_inputs)

    deberta_row_probs = torch.softmax(
        deberta_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 3: RoBERTa inference
    # ------------------------------------------------------

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        roberta_outputs = roberta_model(**roberta_inputs)

    roberta_row_probs = torch.softmax(
        roberta_outputs.logits,
        dim=-1
    ).squeeze(0)


    # ------------------------------------------------------
    # Step 4: Weighted probability ensemble
    # ------------------------------------------------------

    ensemble_probs = (
        0.70 * deberta_row_probs
        + 0.30 * roberta_row_probs
    )


    # ------------------------------------------------------
    # Step 5: Generate Top-3 predictions
    # ------------------------------------------------------

    top3_indices = torch.argsort(
        ensemble_probs,
        descending=True
    )[:3].tolist()

    top3_options = [
        id_to_option[idx]
        for idx in top3_indices
    ]


    # ------------------------------------------------------
    # Step 6: Calculate MAP@3 score for this sample
    # ------------------------------------------------------

    if true_answer in top3_options:

        rank = top3_options.index(true_answer) + 1

        row_score = 1.0 / rank

    else:

        rank = None
        row_score = 0.0


    map3_scores.append(row_score)


    # Store details for analysis
    q10_results.append({
        "true_answer": true_answer,
        "top3_prediction": " ".join(top3_options),
        "correct_rank": rank,
        "map3_score": row_score
    })


# ----------------------------------------------------------
# Step 7: Calculate final average MAP@3
# ----------------------------------------------------------

final_map3 = sum(map3_scores) / len(map3_scores)

rounded_map3 = round(final_map3, 4)


# ----------------------------------------------------------
# Step 8: Create results DataFrame
# ----------------------------------------------------------

q10_results_df = pd.DataFrame(q10_results)


# ----------------------------------------------------------
# Step 9: Display results
# ----------------------------------------------------------

print("\nMAP@3 Results Preview:")

print(q10_results_df.head(10))


print("\nScore Distribution:")

print(
    q10_results_df["map3_score"].value_counts().sort_index(
        ascending=False
    )
)


print("\nFinal MAP@3 :", final_map3)
print("Rounded MAP@3 :", rounded_map3)

Computing MAP@3: 100%|██████████| 100/100 [00:40<00:00,  2.47it/s]


MAP@3 Results Preview:
  true_answer top3_prediction  correct_rank  map3_score
0           B           E A D           NaN         0.0
1           A           E A D           2.0         0.5
2           C           E A D           NaN         0.0
3           B           E A D           NaN         0.0
4           A           E A D           2.0         0.5
5           C           E A D           NaN         0.0
6           E           E A D           1.0         1.0
7           A           E A D           2.0         0.5
8           A           E A D           2.0         0.5
9           A           E A D           2.0         0.5

Score Distribution:
map3_score
1.000000    16
0.500000    17
0.333333    14
0.000000    53
Name: count, dtype: int64

Final MAP@3 : 0.2916666666666667
Rounded MAP@3 : 0.2917


### Conclusion

The final MAP@3 score of the weighted ensemble on the first 100 validation samples is:

`0.2917`

This score was calculated by assigning:

- 1.0 for a correct answer at Rank 1
- 0.5 for a correct answer at Rank 2
- 1/3 for a correct answer at Rank 3
- 0.0 when the correct answer was outside the Top-3